Statistikoj laŭ la plej simpla du-litera silabaro. Ĉiuj vortoj estas dividataj je silaboj el du literoj. 
Rigardu du variantojn: 

- antaŭ-vokala konsonanto: ĉiuj silaboj estas **KV**
- post-vokala konsonanto: ĉiuj silaboj estas **VK**

Helpantaj konsonanto kaj vokalo:
- se mankas vokalo por formi silabon, ni aldonu neprononceblan vokalon `y` 
  - se tio estas komence de vorto oni marku tion `<y`
  - se fine de vorto: `y>`
- se mankas konsonanto por formi silabon, ni aldonu neprononceblan konsonanton `x`
  - se tio estas komence de vorto oni marku tion `<x`
  - se fine de vorto: `x>`

Ekzemploj:

vorto | KV | VK
-----|-----|-----
la   | LA  | <yL Ax>
al   | <xA Ly>  | AL
iu   | <xI xU | Ix Ux>
alta | <xA Ly TA | AL yT Ax>
mem  | ME My> | <yM EM
aero | <xA xE RO | Ax ER Ox>
trans | Ty RA Ny Sy> | <yT yR AN ys
gemaljuneguletoj | GE MA Ly JU NE GU LE TO Jy> | <yG EM AL yJ UN EG UL ET OJ
babiletemulegoj | BA BI LE TE MU LE GO Jy> | <yB AB IL ET EM UL EG OJ

Iuj vortoj estas pli longaj uzante KV, alia male estas pli longaj uzante VK. Iuj vortoj havas la saman kvanton da silaboj en la ambaŭ variantoj. 
La varianto *KV* estas pli oportuna por skribi prefiksojn kaj radikojn.
La varianto *VK* estas pli oportuna por skribi sufiksojn kaj finaĵojn.

Esperanto havas 
- 5 vokalojn: `a, e, i, o, u` 
- kaj 23 konsonantojn: `b, c, ĉ, d, f, g, ĝ, h, ĥ, j, ĵ, k, l, m, n, p, r, s, ŝ, t, ŭ, v, z`

Do en ambaŭ variantoj povas esti teorie:
- 23*5: KV aŭ VK - realaj paroj
- 5: xV aŭ Vx - vokaloj sen konsonanto
- 23: Ky aŭ yK - konsonantoj sen vokalo

Sume _143 = 23\*5 + 5 + 23 = 24*6 - 1_ (minus neebla kombino `xy`). 

Ni dividu la tekston je silaboj per ambaŭ variantoj kaj komparu kiom da diversaj duliteraj silaboj reale aperas en Esperantaj tekstoj.


In [1]:
# constants and imports

from iloj.du_litera_silab_ilo import DuLiteraSilabiIlo, Silabo

from pathlib import Path

# g_base_filename: str - Base filename (without extension) for input and output files
g_base_filename = "all_words"

# input_dir: pathlib.Path - Directory containing input YAML files
g_input_dir = Path("./data/input")

# g_output_dir: pathlib.Path - Output directory for all generated CSV files
g_output_dir = Path("./data/output/dulitera")


In [2]:
# Parse custom YAML format (word: frequency) into dict; avoids yaml library's false-conversion bug
def read_flat_yml(input_file):
    with open(input_file, "r") as f:
        lines = f.readlines()

    all_data = {}
    for line in lines:
        vorto, kvanto = line.strip().split(": ")
        all_data[vorto] = int(kvanto)

    return all_data


In [3]:
# Load input data from YAML file and display basic statistics

# g_all_words: dict[str, int] - Mapping of Esperanto words to their occurrence frequencies
g_all_words = read_flat_yml(str(g_input_dir / f"{g_base_filename}.yaml"))
print("Length:", len(g_all_words))
print("First 10 items:", list(g_all_words.items())[:10])

Length: 49463
First 10 items: [('la', 612479), ('de', 308999), ('kaj', 219359), ('en', 146802), ('al', 96398), ('estas', 71030), ('ne', 68672), ('mi', 57988), ('por', 57813), ('li', 56887)]


In [4]:
# Write syllable breakdown of each word to CSV file

def skribu_silabojn_csv(rezultoj: list, output_file: str, rendering_func) -> None:
    """
    Writes the list of triples (word, silaboj, frequency) as CSV into output_file.
    Columns: vorto,silaboj,frekvenco
    
    Args:
        rezultoj: list of (word, syllables, frequency) tuples
        output_file: path to output CSV file
        rendering_func: function that takes a list of Silabo and returns formatted string
    """
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("vorto,silaboj,frekvenco\n")
        for vorto, silaboj, frekvenco in rezultoj:
            formatigita = rendering_func(silaboj)
            f.write(f"{vorto},{formatigita},{frekvenco}\n")
    print(f"Skribis {len(rezultoj)} vortojn en '{output_file}'")

In [5]:
# Parse all words into syllable structures
def dividu_je_silabojn(all_words: dict, dividilo) -> list:
    """
    Given all_words dict (word -> frequency),
    for each word calls the dividilo function and builds a list of triples:
    (word, array of silaboj, frequency).
    The list preserves the input order.
    
    Args:
        all_words: dict[str, int] - mapping of words to frequencies
        dividilo: function that takes a word string and returns list of Silabo
    
    Returns:
        list of (word, syllables, frequency) tuples
    """
    rezultoj = []
    for vorto, frekvenco in all_words.items():
        silaboj = dividilo(vorto)
        rezultoj.append((vorto, silaboj, frekvenco))
    return rezultoj

In [6]:
# Execute main pipeline: divide all words into AVK and PVK syllables, export both to CSV
ilo = DuLiteraSilabiIlo()
# g_words_with_syllables_avk: list[tuple] - List of (word, syllables_array, frequency) tuples with complete AVK syllable breakdown
g_words_with_syllables_avk = dividu_je_silabojn(g_all_words, ilo.dividu_laŭ_avk)
skribu_silabojn_csv(g_words_with_syllables_avk,
                    str(g_output_dir / f"{g_base_filename}_avk.csv"),
                    ilo.skribu_avk)
# g_words_with_syllables_pvk: list[tuple] - List of (word, syllables_array, frequency) tuples with complete PVK syllable breakdown
g_words_with_syllables_pvk = dividu_je_silabojn(g_all_words, ilo.dividu_laŭ_pvk)
skribu_silabojn_csv(g_words_with_syllables_pvk,
                    str(g_output_dir / f"{g_base_filename}_pvk.csv"),
                    ilo.skribu_pvk)

Skribis 49463 vortojn en 'data/output/dulitera/all_words_avk.csv'
Skribis 49463 vortojn en 'data/output/dulitera/all_words_pvk.csv'


In [7]:
# Calculate distinct syllables and their frequencies
from iloj.du_litera_silab_ilo import (
    NUL_KO_FIN,
    NUL_KO_KOM,
    NUL_VO,
    NUL_KO,
    NUL_VO_FIN,
    NUL_VO_KOM,
)


def kalkulu_distinktajn_silabojn(vortoj_kaj_silaboj: list) -> dict:
    """
    Given a list of triples (vorto, vorto_silaboj, frekvenco),
    returns a dict mapping each distinct syllable (Silabo tuple) to its total frequency.
    A syllable appears as many times as the word it belongs to was used.
    """
    result = {}
    for _, vorto_silaboj, frekvenco in vortoj_kaj_silaboj:
        for silabo in vorto_silaboj:
            result[silabo] = result.get(silabo, 0) + frekvenco
    return result


def kunigu_silabon(silabo: Silabo) -> Silabo:
    if silabo.k == NUL_KO_KOM or silabo.k == NUL_KO_FIN:
        return Silabo(NUL_KO, silabo.v)
    if silabo.v == NUL_VO_KOM or silabo.v == NUL_VO_FIN:
        return Silabo(silabo.k, NUL_VO)
    return silabo


def kalkulu_distinktajn_silabojn_kunige(vortoj_kaj_silaboj: list) -> dict:
    """
    Given a list of triples (vorto, vorto_silaboj, frekvenco),
    returns a dict mapping each distinct syllable (Silabo tuple) to its total frequency.
    """
    result = {}
    for _, vorto_silaboj, frekvenco in vortoj_kaj_silaboj:
        for silabo in vorto_silaboj:
            kunigita_silabo = kunigu_silabon(silabo)
            result[kunigita_silabo] = result.get(kunigita_silabo, 0) + frekvenco
    return result

In [8]:
# Write distinct syllables to CSV with cumulative percentage milestones

def transform_silaboj_to_strings(silaboj_dict: dict, rendering_func) -> dict:
    """
    Transform a dict with Silabo keys (immutable NamedTuples) to a dict with string keys.
    
    Args:
        silaboj_dict: dict[Silabo, int] - mapping from Silabo tuples to frequencies
        rendering_func: function that takes a Silabo and returns string representation
    
    Returns:
        dict[str, int] - mapping from string representations to frequencies
    """
    result = {}
    for silabo, freq in silaboj_dict.items():
        key = rendering_func(silabo)
        result[key] = freq
    return result


def write_syllables_to_csv(dist, output_file):
    total_sil_freq = sum(dist.values())
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("silabo,frekvenco,procento\n")
        cumulative = 0
        next_threshold = 10
        index = 0
        for sil, freq in sorted(dist.items(), key=lambda x: x[1], reverse=True):
            pct = freq / total_sil_freq * 100
            index += 1
            f.write(f"{index} {sil},{freq},{pct:.4f}\n")
            cumulative += pct
            while next_threshold <= 90 and cumulative >= next_threshold:
                f.write(
                    f"# --- ĉi tiuj silaboj kune kovras {cumulative:.4f}% (sojlo: {next_threshold}%) ---\n"
                )
                next_threshold += 10
    print(f"Skribis {len(dist)} distinktajn silabojn en '{output_file}'")


ilo = DuLiteraSilabiIlo()

# distinktaj_silaboj_avk_raw: dict[Silabo, int] - Distinct syllables as immutable tuples → frequencies
distinktaj_silaboj_avk_raw = kalkulu_distinktajn_silabojn(g_words_with_syllables_avk)

# Transform Silabo keys to string keys for CSV output
distinktaj_silaboj_avk = transform_silaboj_to_strings(distinktaj_silaboj_avk_raw, ilo.skribu_silabon_avk)

write_syllables_to_csv(
    distinktaj_silaboj_avk,
    str(g_output_dir / f"{g_base_filename}_distinktaj_silaboj_avk.csv"),
)

# distinktaj_silaboj_pvk_raw: dict[Silabo, int] - Distinct syllables as immutable tuples → frequencies
distinktaj_silaboj_pvk_raw = kalkulu_distinktajn_silabojn(g_words_with_syllables_pvk)

# Transform Silabo keys to string keys for CSV output
distinktaj_silaboj_pvk = transform_silaboj_to_strings(distinktaj_silaboj_pvk_raw, ilo.skribu_silabon_pvk)

write_syllables_to_csv(
    distinktaj_silaboj_pvk,
    str(g_output_dir / f"{g_base_filename}_distinktaj_silaboj_pvk.csv"),
)

Skribis 162 distinktajn silabojn en 'data/output/dulitera/all_words_distinktaj_silaboj_avk.csv'
Skribis 166 distinktajn silabojn en 'data/output/dulitera/all_words_distinktaj_silaboj_pvk.csv'


In [9]:
# Merge positional variants (<x/x>, <y/y>) into unified symbols, write combined syllable frequencies to CSV

ilo = DuLiteraSilabiIlo()

# kunigitaj_silaboj_avk_raw: dict[Silabo, int] - Combined syllables as immutable tuples → frequencies
kunigitaj_silaboj_avk_raw = kalkulu_distinktajn_silabojn_kunige(g_words_with_syllables_avk)

# Transform Silabo keys to string keys for CSV output
kunigitaj_silaboj_avk = transform_silaboj_to_strings(kunigitaj_silaboj_avk_raw, ilo.skribu_silabon_avk)

write_syllables_to_csv(
    kunigitaj_silaboj_avk,
    str(g_output_dir / f"{g_base_filename}_kunigitaj_silaboj_avk.csv"),
)

# kunigitaj_silaboj_pvk_raw: dict[Silabo, int] - Combined syllables as immutable tuples → frequencies
kunigitaj_silaboj_pvk_raw = kalkulu_distinktajn_silabojn_kunige(g_words_with_syllables_pvk)

# Transform Silabo keys to string keys for CSV output
kunigitaj_silaboj_pvk = transform_silaboj_to_strings(kunigitaj_silaboj_pvk_raw, ilo.skribu_silabon_pvk)

write_syllables_to_csv(
    kunigitaj_silaboj_pvk,
    str(g_output_dir / f"{g_base_filename}_kunigitaj_silaboj_pvk.csv"),
)

print(
    "Keys in kunigitaj_silaboj_avk sorted by frequency desc:\n",
    ", ".join([key for key, freq in sorted(kunigitaj_silaboj_avk.items(), key=lambda x: x[1], reverse=True)]),
)


print(
    "Keys in kunigitaj_silaboj_pvk sorted by frequency desc:\n",
    ", ".join([key for key, freq in sorted(kunigitaj_silaboj_pvk.items(), key=lambda x: x[1], reverse=True)]),
)

Skribis 143 distinktajn silabojn en 'data/output/dulitera/all_words_kunigitaj_silaboj_avk.csv'
Skribis 139 distinktajn silabojn en 'data/output/dulitera/all_words_kunigitaj_silaboj_pvk.csv'
Keys in kunigitaj_silaboj_avk sorted by frequency desc:
 Ny, Sy, Jy, LA, xA, xE, Ry, DE, TA, KA, Ly, LI, TI, xO, TO, RO, Py, RA, RI, Ky, RE, TE, KO, xU, xI, PO, NI, NO, KI, Ty, MA, VI, LO, MI, DO, My, NE, Ŭy, MO, PE, NA, DI, SI, SE, CI, LE, DA, VA, ME, PA, By, Gy, KU, VE, NU, ĜI, VO, TU, KE, SO, GI, GA, SU, ĈI, FA, CE, DU, SA, FO, BO, Dy, FI, CO, ZI, RU, JA, Fy, FE, GO, LU, ĜO, JO, MU, ZO, ĈE, ŜI, BA, ZA, JE, HO, BE, PI, ĈA, HA, ĜA, GE, ZE, ĴO, JU, PU, BI, CA, ŜA, ĈU, ĜE, HE, Ŝy, BU, VU, Ĉy, FU, GU, HI, ĈO, ĜU, Vy, ZU, ŜO, ŬE, ĴU, ŬA, HU, ŜU, CU, ĴE, Zy, Cy, ŜE, ŬI, ĤO, ĤA, JI, ĤI, ŬU, Ĝy, ĴA, Ĥy, ĤE, ŬO, Hy, ĤU, ĴI, Ĵy
Keys in kunigitaj_silaboj_pvk sorted by frequency desc:
 Ix, yT, Ax, yL, Ox, yK, Ex, yD, yR, yP, yS, ON, AJ, EN, yN, AN, IS, OJ, yM, AS, ER, AL, ES, IN, yV, OR, AR, Ux, yF, EL, UN, A

In [10]:
# Visual comparison of AVK and PVK syllable distributions

VOKALOJ = ["a", "e", "i", "o", "u"]
KONSONANTOJ = ["b", "c", "ĉ", "d", "f", "g", "ĝ", "h", "ĥ", "j", "ĵ", "k", "l", "m", "n", "p", "r", "s", "ŝ", "t", "ŭ", "v", "z"]

def generiĝi_ĉiuj_silaboj():
    """Generate all 143 valid Silabo combinations:
    - 5 xV (vowel without consonant)
    - 23 Ky (consonant without vowel)
    - 115 KV (all consonant-vowel combinations)
    """
    silaboj = []
    
    # xV: vowel without consonant
    for v in VOKALOJ:
        silaboj.append(Silabo(k="x", v=v))
    
    # Ky: consonant without vowel
    for k in KONSONANTOJ:
        silaboj.append(Silabo(k=k, v="y"))
    
    # KV: all consonant-vowel pairs
    for k in KONSONANTOJ:
        for v in VOKALOJ:
            silaboj.append(Silabo(k=k, v=v))
    
    return silaboj


def kompari_silabojn_avk_vs_pvk(dist_avk: dict, dist_pvk: dict, output_file: str):
    """Compare AVK and PVK syllable distributions.
    
    Args:
        dist_avk: dict[str, int] - syllables in AVK format with frequencies
        dist_pvk: dict[str, int] - syllables in PVK format with frequencies
        output_file: path to output CSV file
    """
    # Generate all valid syllables
    ĉiuj_silaboj = generiĝi_ĉiuj_silaboj()
    
    # Calculate totals
    total_avk = sum(dist_avk.values())
    total_pvk = sum(dist_pvk.values())
    
    # Build comparison rows
    rows = []
    for silabo in ĉiuj_silaboj:
        silabo_avk_str = ilo.skribu_silabon_avk(silabo)
        silabo_pvk_str = ilo.skribu_silabon_pvk(silabo)
        
        # Check if syllable exists in AVK dict
        if silabo_avk_str in dist_avk:
            count_avk = dist_avk[silabo_avk_str]
            pct_avk = f"{count_avk / total_avk * 100:.4f}%"
        else:
            count_avk = "--"
            pct_avk = "--"
        
        # Check if syllable exists in PVK dict
        if silabo_pvk_str in dist_pvk:
            count_pvk = dist_pvk[silabo_pvk_str]
            pct_pvk = f"{count_pvk / total_pvk * 100:.4f}%"
        else:
            count_pvk = "--"
            pct_pvk = "--"
        
        rows.append((silabo_avk_str, count_avk, pct_avk, silabo_pvk_str, count_pvk, pct_pvk))
    
    # Sort by AVK syllable string
    rows.sort(key=lambda x: x[0])
    
    # Write CSV
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("silabo_avk,count_avk,percent_avk,silabo_pvk,count_pvk,percent_pvk\n")
        for silabo_avk, count_avk, pct_avk, silabo_pvk, count_pvk, pct_pvk in rows:
            f.write(f"{silabo_avk},{count_avk},{pct_avk},{silabo_pvk},{count_pvk},{pct_pvk}\n")
    
    print(f"Skribis {len(rows)} silabojn en '{output_file}'")


# Generate comparison
kompari_silabojn_avk_vs_pvk(
    distinktaj_silaboj_avk,
    distinktaj_silaboj_pvk,
    str(g_output_dir / f"{g_base_filename}_komparo_avk_pvk.csv")
)

Skribis 143 silabojn en 'data/output/dulitera/all_words_komparo_avk_pvk.csv'


In [11]:
# Comparison of INCOMPLETE syllables only (neplenaj silaboj: k="x" or v="y")

def generiĝi_neplenajn_silabojn():
    """Generate only incomplete Silabo combinations (where k="x" or v="y"):
    - 5 xV (vowel without consonant)
    - 23 Ky (consonant without vowel)
    Total: 28 incomplete syllables
    """
    silaboj = []
    
    # xV: vowel without consonant
    for v in VOKALOJ:
        silaboj.append(Silabo(k="x", v=v))
    
    # Ky: consonant without vowel
    for k in KONSONANTOJ:
        silaboj.append(Silabo(k=k, v="y"))
    
    return silaboj


def generiĝi_neplenajn_kom_kaj_fin_silabojn():
    """Generate incomplete syllables categorized by position (start or finish):
    Start-incomplete (kom): k="x" → 5 syllables (xV)
    End-incomplete (fin): v="y" → 23 syllables (Ky)
    
    Returns:
        list of tuples: (silabo, position) where position is "kom" (start) or "fin" (end)
    """
    silaboj = []
    
    for v in VOKALOJ:
        silaboj.append(Silabo(k="<x", v=v))
        silaboj.append(Silabo(k="x>", v=v))
    
    # End-incomplete: Ky (v="y" means no vowel at end)
    for k in KONSONANTOJ:
        silaboj.append(Silabo(k=k, v="y>"))
        silaboj.append(Silabo(k=k, v="<y"))
    
    return silaboj


def kompari_neplenajn_silabojn_avk_vs_pvk(dist_avk: dict, dist_pvk: dict, output_file: str):
    """Compare AVK and PVK distributions for incomplete syllables only.
    
    Args:
        dist_avk: dict[str, int] - syllables in AVK format with frequencies
        dist_pvk: dict[str, int] - syllables in PVK format with frequencies
        output_file: path to output CSV file
    """
    # Generate only incomplete syllables
    neplenaj_silaboj = generiĝi_neplenajn_silabojn()
    
    # Calculate totals
    total_avk = sum(dist_avk.values())
    total_pvk = sum(dist_pvk.values())
    
    # Build comparison rows
    rows = []
    for silabo in neplenaj_silaboj:
        silabo_avk_str = ilo.skribu_silabon_avk(silabo)
        silabo_pvk_str = ilo.skribu_silabon_pvk(silabo)
        
        # Check if syllable exists in AVK dict
        if silabo_avk_str in dist_avk:
            count_avk = dist_avk[silabo_avk_str]
            pct_avk = f"{count_avk / total_avk * 100:.4f}%"
        else:
            count_avk = "--"
            pct_avk = "--"
        
        # Check if syllable exists in PVK dict
        if silabo_pvk_str in dist_pvk:
            count_pvk = dist_pvk[silabo_pvk_str]
            pct_pvk = f"{count_pvk / total_pvk * 100:.4f}%"
        else:
            count_pvk = "--"
            pct_pvk = "--"
        
        rows.append((silabo_avk_str, count_avk, pct_avk, silabo_pvk_str, count_pvk, pct_pvk))
    
    # Sort by AVK syllable string
    rows.sort(key=lambda x: x[0])
    
    # Write CSV
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("silabo_avk,count_avk,percent_avk,silabo_pvk,count_pvk,percent_pvk\n")
        for silabo_avk, count_avk, pct_avk, silabo_pvk, count_pvk, pct_pvk in rows:
            f.write(f"{silabo_avk},{count_avk},{pct_avk},{silabo_pvk},{count_pvk},{pct_pvk}\n")
    
    print(f"Skribis {len(rows)} neplenajn silabojn en '{output_file}'")


def kompari_neplenajn_kom_kaj_fin_silabojn_avk_vs_pvk(dist_avk: dict, dist_pvk: dict, output_file: str):
    """Compare AVK and PVK distributions for incomplete syllables, categorized by position (start or end).
    Writes vertically: AVK syllables, blank line, then PVK syllables.
    
    Args:
        dist_avk: dict[str, int] - syllables in AVK format with frequencies
        dist_pvk: dict[str, int] - syllables in PVK format with frequencies
        output_file: path to output CSV file
    """
    # Generate incomplete syllables with position info
    neplenaj_silaboj = generiĝi_neplenajn_kom_kaj_fin_silabojn()
    
    # Calculate totals
    total_avk = sum(dist_avk.values())
    total_pvk = sum(dist_pvk.values())
    
    # Build AVK and PVK rows separately
    avk_rows = []
    pvk_rows = []
    
    for silabo in neplenaj_silaboj:
        silabo_avk_str = ilo.skribu_silabon_avk(silabo)
        silabo_pvk_str = ilo.skribu_silabon_pvk(silabo)
        
        # Check if syllable exists in AVK dict - only add if found
        if silabo_avk_str in dist_avk:
            count_avk = dist_avk[silabo_avk_str]
            pct_avk = f"{count_avk / total_avk * 100:.4f}%"
            avk_rows.append((silabo_avk_str, count_avk, pct_avk))
        
        # Check if syllable exists in PVK dict - only add if found
        if silabo_pvk_str in dist_pvk:
            count_pvk = dist_pvk[silabo_pvk_str]
            pct_pvk = f"{count_pvk / total_pvk * 100:.4f}%"
            pvk_rows.append((silabo_pvk_str, count_pvk, pct_pvk))
    
    # Sort both rows by syllable string
    avk_rows.sort(key=lambda x: x[0])
    pvk_rows.sort(key=lambda x: x[0])
    
    # Write CSV with vertical layout: AVK section, blank line, PVK section
    with open(output_file, "w", encoding="utf-8") as f:
        # Write AVK section
        f.write("silabo,count,percent\n")
        for silabo, count, pct in avk_rows:
            f.write(f"{silabo},{count},{pct}\n")
        
        # Blank line separator
        f.write("\n")
        
        # Write PVK section
        f.write("silabo,count,percent\n")
        for silabo, count, pct in pvk_rows:
            f.write(f"{silabo},{count},{pct}\n")

    print(f"Skribis {len(avk_rows)} AVK + {len(pvk_rows)} PVK neplenajn silabojn en '{output_file}'")


# Generate comparison for incomplete syllables only
kompari_neplenajn_silabojn_avk_vs_pvk(
    distinktaj_silaboj_avk,
    distinktaj_silaboj_pvk,
    str(g_output_dir / f"{g_base_filename}_komparo_neplenaj_avk_pvk.csv")
)

# Generate comparison for incomplete syllables categorized by position
kompari_neplenajn_kom_kaj_fin_silabojn_avk_vs_pvk(
    distinktaj_silaboj_avk,
    distinktaj_silaboj_pvk,
    str(g_output_dir / f"{g_base_filename}_komparo_neplenaj_kom_kaj_fin_avk_pvk.csv")
)

Skribis 28 neplenajn silabojn en 'data/output/dulitera/all_words_komparo_neplenaj_avk_pvk.csv'
Skribis 19 AVK + 27 PVK neplenajn silabojn en 'data/output/dulitera/all_words_komparo_neplenaj_kom_kaj_fin_avk_pvk.csv'


In [12]:
# Count incomplete syllables (x/y placeholders) by type in both AVK and PVK sets

def sum_matching_syllables_raw(silaboj_raw: dict, predicate) -> int:
    """Sum frequencies of syllables matching the predicate.
    
    Args:
        silaboj_raw: dict[Silabo, int] - syllables with frequencies (Silabo as key)
        predicate: function that takes a Silabo and returns True if it matches
    
    Returns:
        int - sum of frequencies for matching syllables
    """
    total = 0
    for silabo, freq in silaboj_raw.items():
        if predicate(silabo):
            total += freq
    return total


# AVK set: separate each case with descriptions
avk_x = sum_matching_syllables_raw(distinktaj_silaboj_avk_raw, lambda s: s.k == 'x')
avk_x_left = sum_matching_syllables_raw(distinktaj_silaboj_avk_raw, lambda s: s.k == '<x')
avk_y = sum_matching_syllables_raw(distinktaj_silaboj_avk_raw, lambda s: s.v == 'y')
avk_y_right = sum_matching_syllables_raw(distinktaj_silaboj_avk_raw, lambda s: s.v == 'y>')
avk_total = avk_x + avk_x_left + avk_y + avk_y_right

print("AVK set:")
print(f"  Syllables with x (vowel cluster): {avk_x}")
print(f"  Syllables with <x (vowel at word start): {avk_x_left}")
print(f"  Syllables with y (consonant cluster): {avk_y}")
print(f"  Syllables with y> (consonant at word end): {avk_y_right}")
print(f"  Total: {avk_total}")

# PVK set: separate each case with descriptions
pvk_x = sum_matching_syllables_raw(distinktaj_silaboj_pvk_raw, lambda s: s.k == 'x')
pvk_x_right = sum_matching_syllables_raw(distinktaj_silaboj_pvk_raw, lambda s: s.k == 'x>')
pvk_y = sum_matching_syllables_raw(distinktaj_silaboj_pvk_raw, lambda s: s.v == 'y')
pvk_y_left = sum_matching_syllables_raw(distinktaj_silaboj_pvk_raw, lambda s: s.v == '<y')
pvk_total = pvk_x + pvk_x_right + pvk_y + pvk_y_left

print("\nPVK set:")
print(f"  Syllables with x (vowel cluster): {pvk_x}")
print(f"  Syllables with x> (vowel at word end): {pvk_x_right}")
print(f"  Syllables with y (consonant cluster): {pvk_y}")
print(f"  Syllables with <y (word starting from consonant): {pvk_y_left}")
print(f"  Total: {pvk_total}")


AVK set:
  Syllables with x (vowel cluster): 818872
  Syllables with <x (vowel at word start): 1233596
  Syllables with y (consonant cluster): 3078008
  Syllables with y> (consonant at word end): 2743350
  Total: 7873826

PVK set:
  Syllables with x (vowel cluster): 818872
  Syllables with x> (vowel at word end): 3306868
  Syllables with y (consonant cluster): 3078008
  Syllables with <y (word starting from consonant): 4816622
  Total: 12020370


In [13]:
# Analyze word boundaries: count words starting/ending with vowels vs consonants
vowels = "aeiou"
consonants = "bcĉdfgĝhĥjĵklmnprsŝtŭvz"

start_vowel_count = 0
start_consonant_count = 0
end_vowel_count = 0
end_consonant_count = 0

fr_start_vowel_count = 0
fr_start_consonant_count = 0
fr_end_vowel_count = 0
fr_end_consonant_count = 0

for word in g_all_words:
    if word[0] in vowels:
        start_vowel_count += 1
        fr_start_vowel_count += g_all_words[word]
    elif word[0] in consonants:
        start_consonant_count += 1
        fr_start_consonant_count += g_all_words[word]
    if word[-1] in vowels:
        end_vowel_count += 1
        fr_end_vowel_count += g_all_words[word]
    elif word[-1] in consonants:
        end_consonant_count += 1
        fr_end_consonant_count += g_all_words[word]

total_words = len(g_all_words)
fr_total_words = sum(g_all_words.values())

print(f"Total words: {total_words} (frequency sum: {fr_total_words})")
print(f"Words starting with a vowel: {start_vowel_count} {start_vowel_count / total_words * 100:.2f}% (frequency sum: {fr_start_vowel_count}) ({fr_start_vowel_count / fr_total_words * 100:.2f}%)")
print(f"Words starting with a consonant: {start_consonant_count} {start_consonant_count / total_words * 100:.2f}% (frequency sum: {fr_start_consonant_count}) ({fr_start_consonant_count / fr_total_words * 100:.2f}%)")
print(f"Words ending with a vowel: {end_vowel_count} {end_vowel_count / total_words * 100:.2f}% (frequency sum: {fr_end_vowel_count}) ({fr_end_vowel_count / fr_total_words * 100:.2f}%)")
print(f"Words ending with a consonant: {end_consonant_count} {end_consonant_count / total_words * 100:.2f}% (frequency sum: {fr_end_consonant_count}) ({fr_end_consonant_count / fr_total_words * 100:.2f}%)")


Total words: 49463 (frequency sum: 6050218)
Words starting with a vowel: 10110 20.44% (frequency sum: 1233596) (20.39%)
Words starting with a consonant: 39353 79.56% (frequency sum: 4816622) (79.61%)
Words ending with a vowel: 23317 47.14% (frequency sum: 3306868) (54.66%)
Words ending with a consonant: 26146 52.86% (frequency sum: 2743350) (45.34%)


In [14]:
# Compare syllable counts between AVK and PVK for each word

# Create dictionaries for quick lookup by word
avk_dict = {vorto: silaboj for vorto, silaboj, _ in g_words_with_syllables_avk}
pvk_dict = {vorto: silaboj for vorto, silaboj, _ in g_words_with_syllables_pvk}

print("Vortoj kun silabar-komparo (AVK vs PVK):")
print("vorto,avk_count,pvk_count")

different_count = 0
avk_total = 0
pvk_total = 0
pvk_gajnas = 0

for vorto in g_all_words:
    avk_count = len(avk_dict.get(vorto, []))
    pvk_count = len(pvk_dict.get(vorto, []))
    
    # Skip words where syllable counts are equal
    if avk_count == pvk_count:
        continue
    
    # Count different pairs and accumulate syllable counts
    different_count += 1
    avk_total += avk_count
    pvk_total += pvk_count
    
    if pvk_count < avk_count:
        pvk_gajnas += 1
    
print(f"\nDiferaj paroj: {different_count}")
print(f"AVK silaboj-totalo: {avk_total}")
print(f"PVK silaboj-totalo: {pvk_total}")
print(f"PVK gajnas en {pvk_gajnas} kazoj")
total_syllables_avk = sum(len(silaboj) * frekvenco for vorto, silaboj, frekvenco in g_words_with_syllables_avk)
print(f"Sumo de ĉiuj silaboj per AVK: {total_syllables_avk}")
total_syllables_pvk = sum(len(silaboj) * frekvenco for vorto, silaboj, frekvenco in g_words_with_syllables_pvk)
print(f"Sumo de ĉiuj silaboj per PVK: {total_syllables_pvk}")
print(f"diferenco {total_syllables_pvk - total_syllables_avk} silaboj (PVK - AVK)")


Vortoj kun silabar-komparo (AVK vs PVK):
vorto,avk_count,pvk_count

Diferaj paroj: 24161
AVK silaboj-totalo: 118279
PVK silaboj-totalo: 131486
PVK gajnas en 5477 kazoj
Sumo de ĉiuj silaboj per AVK: 18359344
Sumo de ĉiuj silaboj per PVK: 20432616
diferenco 2073272 silaboj (PVK - AVK)
